# BSOS anxiety measurement audit

Secondary-only analysis of the supplied file. This notebook cannot test personalisation, comparison mediation or literacy moderation. Do not merge unrelated respondents. The guide’s 1–4 statement conflicts with the verified 0–3 item codes. No raw participant records are exported.

In [1]:
from pathlib import Path
import hashlib
source=Path("../upload/mental_health2425v8in.csv")
if not source.exists():
    from google.colab import files
    uploaded=files.upload()
    candidates=[Path(n) for n in uploaded if n.lower().endswith(".csv")]
    assert len(candidates)==1, "Upload exactly the original BSOS CSV"
    source=candidates[0]
assert hashlib.sha256(source.read_bytes()).hexdigest()=="f02595937001d533277f3d0a851069209a9bc0aabee46224ad3f52dcf22e9eb8", "File version differs: review rather than bypass"

In [2]:
from pathlib import Path
import hashlib,json,platform
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import scipy
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def analyse(path,output):
    path=Path(path);out=Path(output);out.mkdir(parents=True,exist_ok=True)
    d=pd.read_csv(path)
    mapping={'Not at all':0,'Several days':1,'More than half of the days':2,'Nearly every day':3}
    gad=d[[f'gad7_{i}' for i in range(1,8)]].apply(lambda x:x.str.strip().map(mapping))
    phq=d[[f'phq9_{i}' for i in range(1,10)]].apply(lambda x:x.str.strip().map(mapping))
    assert gad.notna().all().all() and phq.notna().all().all(),'Unknown/missing response; review source before scoring'
    gc=d[[f'gad7_{i}NUM' for i in range(1,8)]].to_numpy()
    pc=d[[f'phq9_{i}NUM' if i!=6 else 'phq9_6 NUM' for i in range(1,10)]].to_numpy()
    assert np.array_equal(gc,gad.to_numpy()) and np.array_equal(pc,phq.to_numpy())
    g=gad.sum(axis=1);p=phq.sum(axis=1)
    assert (g==d.gad7_COMP).all() and (p==d.phq9_COMP).all()
    bands=pd.cut(g,[-1,4,9,14,21],labels=['minimal anxiety','mild anxiety','moderate anxiety','severe anxiety'])
    assert (bands.astype(str)==d.gad7_severity.str.strip()).all()
    def alpha(a):
        return a.shape[1]/(a.shape[1]-1)*(1-a.var(axis=0,ddof=1).sum()/a.sum(axis=1).var(ddof=1))
    rng=np.random.default_rng(841206);rows=[];summaries={}
    for label,mask in [('All supplied records',np.ones(len(d),dtype=bool)),('Birth years 1997 to 2012',d.year_1.between(1997,2012).to_numpy())]:
        ga=gad.loc[mask].to_numpy();pa=phq.loc[mask].to_numpy();gs=ga.sum(axis=1);ps=pa.sum(axis=1);n=len(gs)
        ab=[];rb=[]
        for _ in range(2000):
            ix=rng.integers(0,n,n);ab.append(alpha(ga[ix]));rb.append(spearmanr(gs[ix],ps[ix]).statistic)
        rho,pv=spearmanr(gs,ps)
        info={'n':n,'gad_mean':float(gs.mean()),'gad_sd':float(gs.std(ddof=1)),'gad_median':float(np.median(gs)),'gad_q1':float(np.quantile(gs,.25)),'gad_q3':float(np.quantile(gs,.75)),'gad_min':int(gs.min()),'gad_max':int(gs.max()),'gad_alpha':float(alpha(ga)),'alpha_ci':np.quantile(ab,[.025,.975]).tolist(),'gad_floor_n':int((gs==0).sum()),'gad_ceiling_n':int((gs==21).sum()),'gad_ge10_n':int((gs>=10).sum()),'phq_mean':float(ps.mean()),'phq_alpha':float(alpha(pa)),'gad_phq_rho':float(rho),'gad_phq_p':float(pv),'rho_ci':np.quantile(rb,[.025,.975]).tolist()}
        summaries[label]=info
        for band in bands.cat.categories:
            count=int((bands[mask]==band).sum());rows.append({'sample':label,'band':band,'n':count,'denominator':n,'percentage':count/n*100})
    pd.DataFrame(rows).to_csv(out/'gad_severity_counts.csv',index=False)
    corr=[]
    for i in range(7):corr.append({'item':i+1,'corrected_item_total_pearson_r':float(np.corrcoef(gc[:,i],gc.sum(axis=1)-gc[:,i])[0,1])})
    pd.DataFrame(corr).to_csv(out/'gad_item_diagnostics.csv',index=False)
    pd.DataFrame({'column':d.columns,'dtype':d.dtypes.astype(str).values,'missing_n':d.isna().sum().values}).to_csv(out/'column_audit.csv',index=False)
    result={'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'source_rows':len(d),'source_columns':len(d.columns),'exact_duplicate_rows':int(d.duplicated().sum()),'birth_year_min':int(d.year_1.min()),'birth_year_max':int(d.year_1.max()),'states':int(d.state_1.nunique()),'gad_missing_items':int(gad.isna().sum().sum()),'gad_numeric_mismatches':int((gc!=gad.to_numpy()).sum()),'gad_total_mismatches':int((g!=d.gad7_COMP).sum()),'gad_band_mismatches':int((bands.astype(str)!=d.gad7_severity).sum()),'summaries':summaries,'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'scipy':scipy.__version__,'seed':841206,'bootstrap_resamples':2000,'analysis_scope':'Measurement benchmark only. No social media, personalisation, comparison or literacy variables; original H1-H3 not estimable. Collection dates and sampling provenance unresolved.'}
    (out/'bsos_results.json').write_text(json.dumps(result,indent=2))
    fig,ax=plt.subplots(figsize=(7,3.8));counts=pd.Series(g).value_counts().reindex(range(22),fill_value=0);ax.bar(counts.index,counts.values,color='#36596b');ax.set(xlabel='Recomputed GAD-7 total',ylabel='Records',title='Anxiety symptom scores in the supplied file (n = 579)',xticks=[0,5,10,15,21]);ax.spines[['top','right']].set_visible(False);fig.tight_layout();fig.savefig(out/'gad_distribution.png',dpi=180);plt.close(fig)
    print(json.dumps(result,indent=2));return result



In [3]:
result=analyse(source,Path("bsos_revision/results"))

{
  "sha256": "f02595937001d533277f3d0a851069209a9bc0aabee46224ad3f52dcf22e9eb8",
  "source_rows": 579,
  "source_columns": 118,
  "exact_duplicate_rows": 0,
  "birth_year_min": 1971,
  "birth_year_max": 2003,
  "states": 48,
  "gad_missing_items": 0,
  "gad_numeric_mismatches": 0,
  "gad_total_mismatches": 0,
  "gad_band_mismatches": 0,
  "summaries": {
    "All supplied records": {
      "n": 579,
      "gad_mean": 8.324697754749568,
      "gad_sd": 5.685067677865882,
      "gad_median": 7.0,
      "gad_q1": 4.0,
      "gad_q3": 12.0,
      "gad_min": 0,
      "gad_max": 21,
      "gad_alpha": 0.9193163006040714,
      "alpha_ci": [
        0.9070387120221156,
        0.9297699174028938
      ],
      "gad_floor_n": 28,
      "gad_ceiling_n": 21,
      "gad_ge10_n": 216,
      "phq_mean": 9.373056994818652,
      "phq_alpha": 0.8905700936322601,
      "gad_phq_rho": 0.024104888569069328,
      "gad_phq_p": 0.5626886464994628,
      "rho_ci": [
        -0.05668698626179197,
        0.

## Interpretation
The totals and alpha describe the supplied records, not clinical diagnoses or population prevalence. GAD-7–PHQ-9 correlation is exploratory; its weak observed relationship does not prove independence or invalidity. Collection dates and original sampling remain unresolved. The birth-cohort subset is not verified age-at-survey eligibility. Full protocol, discrepancies and original hypothesis limits appear in Appendix J.

To export aggregate outputs in Colab, zip the bsos_revision/results folder; do not include the uploaded participant-level CSV. The notebook reports its actual software versions. Reproduction should match counts exactly and continuous values within rtol=1e-8, atol=1e-10. References: supplied codebook and guide; Spitzer et al. (2006), DOI 10.1001/archinte.166.10.1092; NIH GAD-7 guidance https://www.nih.gov/node/19876.